In [1]:
from google.colab import userdata
from googleapiclient.discovery import build

youtube = build("youtube", "v3", developerKey=userdata.get("YOUTUBE_API_KEY"))

response = youtube.search().list(
    q="skincare routine", type="channel", part="snippet", maxResults=3
).execute()

for item in response["items"]:
    print(item["snippet"]["channelTitle"], "|", item["snippet"]["channelId"])

Hyram | UC2sYit3cZ2CuD_8FHYH7O_Q
Doctorly | UCHCZnC9akNA9pBP7aJGNKdg
Skincare routine  | UCVA7plSVKdNxuqexPv5hn-g


In [2]:
queries = [
    "skincare routine", "makeup tutorial", "fashion haul", "outfit ideas",
    "get ready with me", "beauty product review", "styling tips", "hair care routine",
]

channel_ids = {}
for q in queries:
    response = youtube.search().list(
        q=q, type="channel", part="snippet", maxResults=50
    ).execute()
    for item in response["items"]:
        channel_ids[item["snippet"]["channelId"]] = q

print(len(channel_ids), "unique channels found")

400 unique channels found


In [3]:
import pandas as pd

ids = list(channel_ids.keys())
rows = []
for i in range(0, len(ids), 50):
    batch = ids[i:i+50]
    resp = youtube.channels().list(
        part="snippet,statistics,contentDetails", id=",".join(batch)
    ).execute()
    for ch in resp["items"]:
        stats = ch["statistics"]
        rows.append({
            "name": ch["snippet"]["title"],
            "channel_id": ch["id"],
            "url": f"https://www.youtube.com/channel/{ch['id']}",
            "followers": None if stats.get("hiddenSubscriberCount") else int(stats["subscriberCount"]),
            "video_count": int(stats.get("videoCount", 0)),
            "country": ch["snippet"].get("country", "Not Available"),
            "description": ch["snippet"].get("description", ""),
            "uploads_playlist": ch["contentDetails"]["relatedPlaylists"]["uploads"],
            "found_via": channel_ids[ch["id"]],
        })

df = pd.DataFrame(rows)
print(len(df), "channels collected")
print(df["followers"].between(5000, 100000).sum(), "are in the 5k-100k micro range")
df[["name", "followers", "country", "found_via"]].head(15)

400 channels collected
52 are in the 5k-100k micro range


,name,followers,country,found_via
0,Skincare routine,21,US,skincare routine
1,Korean skincare routine 💫,8,Not Available,skincare routine
2,Skin Care Routine,11900,PK,skincare routine
3,Skin Care Routine,74,PK,skincare routine
4,Skincare routine,8,GB,skincare routine
5,Hailey Rhode Bieber,2760000,US,skincare routine
6,Atomy Beauty Routine Italia,169,IT,skincare routine
7,Minute Munch + Skincare routine,15,Not Available,skincare routine
8,Kitty Skincare Routine,55,Not Available,skincare routine
9,Skincare routine,1,Not Available,skincare routine


In [4]:
micro = df[df["followers"].between(5000, 100000)].copy()

def get_recent_stats(playlist_id, n=10):
    try:
        pl = youtube.playlistItems().list(
            part="contentDetails", playlistId=playlist_id, maxResults=n
        ).execute()
        vids = [it["contentDetails"]["videoId"] for it in pl["items"]]
        if not vids:
            return None
        vr = youtube.videos().list(part="snippet,statistics", id=",".join(vids)).execute()
    except Exception as e:
        print("skipped", playlist_id, "->", str(e)[:80])
        return None

    views_total, interactions_total, used = 0, 0, 0
    titles, dates = [], []
    for v in vr["items"]:
        s = v["statistics"]
        titles.append(v["snippet"]["title"])
        dates.append(v["snippet"]["publishedAt"])
        # skip videos where likes are hidden or there are no views
        if "likeCount" not in s or int(s.get("viewCount", 0)) == 0:
            continue
        views_total += int(s["viewCount"])
        interactions_total += int(s["likeCount"]) + int(s.get("commentCount", 0))
        used += 1

    return {
        "avg_views": round(views_total / used) if used else None,
        "engagement_rate": round(interactions_total / views_total * 100, 2) if views_total else None,
        "videos_used": used,
        "last_upload": max(dates)[:10] if dates else None,
        "recent_titles": titles[:3],
    }

results = []
for _, row in micro.iterrows():
    stats = get_recent_stats(row["uploads_playlist"]) or {}
    results.append({"channel_id": row["channel_id"], **stats})

micro = micro.merge(pd.DataFrame(results), on="channel_id", how="left")
micro[["name", "followers", "engagement_rate", "avg_views", "last_upload", "videos_used"]]

,name,followers,engagement_rate,avg_views,last_upload,videos_used
0,Skin Care Routine,11900,1.53,14097.0,2026-10-03,10
1,My Beauty Routine,5470,1.57,1392.0,2026-05-14,10
2,Rishika,62800,NaN,NaN,2026-10-03,0
3,Notjustarushi,53000,0.29,161522.0,2026-10-02,4
4,Aanyaa,63700,4.11,24575.0,2026-10-02,8
5,Magesh Makeup tutorial,35300,0.93,2093.0,2026-08-16,10
6,Makeup tutorial,5740,1.53,1968.0,2026-10-02,10
7,KryolanOfficial,34800,3.25,870.0,2026-10-01,10
8,Makeup Melanin,96300,4.39,57115.0,2025-12-05,10
9,Vijaya Haul Fashion's,27500,1.55,1120.0,2026-09-21,9


In [5]:
import pandas as pd

# ---------- Rules (change these numbers and re-run; document them in the README) ----------
MIN_FOLLOWERS, MAX_FOLLOWERS = 5000, 100000
MIN_ENGAGEMENT = 1.0        # percent
MAX_ENGAGEMENT_FLAG = 12.0  # above this: still passes, but flagged for manual review
MIN_AVG_VIEWS = 500
MIN_VIDEOS_USED = 5         # engagement must be based on at least 5 videos
MAX_INACTIVE_DAYS = 90

NICHE_KEYWORDS = {
    "Beauty": ["skincare", "skin care", "makeup", "beauty", "hair", "cosmetic", "lipstick", "serum"],
    "Fashion": ["fashion", "outfit", "style", "styling", "haul", "ootd", "lookbook", "wardrobe"],
}
BRAND_WORDS = ["official", "official channel", "® ", "™", " pvt", " ltd"]

# ---------- Merge engagement stats back onto ALL 400 channels ----------
stat_cols = ["channel_id", "avg_views", "engagement_rate", "videos_used", "last_upload", "recent_titles"]
full = df.merge(micro[stat_cols], on="channel_id", how="left")

def classify_niche(row):
    titles = row["recent_titles"] if isinstance(row["recent_titles"], list) else []
    text = f"{row['name']} {row['description']} {' '.join(titles)}".lower()
    scores = {n: sum(k in text for k in kws) for n, kws in NICHE_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "Other"

def evaluate(row):
    reasons, flag = [], ""
    f = row["followers"]
    if pd.isna(f):
        reasons.append("Follower count hidden")
    elif not (MIN_FOLLOWERS <= f <= MAX_FOLLOWERS):
        reasons.append(f"Followers {int(f)} outside {MIN_FOLLOWERS}-{MAX_FOLLOWERS}")

    if reasons:  # engagement was only computed for channels in the follower range
        return reasons, flag

    if pd.isna(row["engagement_rate"]) or row["videos_used"] < MIN_VIDEOS_USED:
        reasons.append("Engagement data unavailable or too few videos")
    else:
        if row["engagement_rate"] < MIN_ENGAGEMENT:
            reasons.append(f"Engagement {row['engagement_rate']}% below {MIN_ENGAGEMENT}%")
        if row["engagement_rate"] > MAX_ENGAGEMENT_FLAG:
            flag = "Unusually high engagement - verify manually"
    if pd.notna(row["avg_views"]) and row["avg_views"] < MIN_AVG_VIEWS:
        reasons.append(f"Avg views {int(row['avg_views'])} below {MIN_AVG_VIEWS}")

    if pd.notna(row["last_upload"]):
        days = (pd.Timestamp.today() - pd.to_datetime(row["last_upload"])).days
        if days > MAX_INACTIVE_DAYS:
            reasons.append(f"Inactive: last upload {days} days ago")

    if row["niche"] == "Other":
        reasons.append("Not fashion/beauty content")
    if any(b in f"{row['name']} {row['description'][:300]}".lower() for b in BRAND_WORDS):
        reasons.append("Looks like a brand/official account")
    return reasons, flag

full["niche"] = full.apply(classify_niche, axis=1)
evals = full.apply(evaluate, axis=1)
full["reasons"] = [r for r, _ in evals]
full["flag"] = [fl for _, fl in evals]
full["status"] = full["reasons"].apply(lambda r: "PASS" if not r else "FAIL")
full["fail_reason"] = full["reasons"].apply(lambda r: "; ".join(r))
full = full.drop(columns=["reasons"])

print(full["status"].value_counts())
print("\nTop fail reasons:")
print(full[full.status == "FAIL"]["fail_reason"].str.split("; ").explode().str.replace(r"\d+", "N", regex=True).value_counts().head(8))

full[full.status == "PASS"][["name", "followers", "engagement_rate", "niche", "last_upload", "flag"]]

status
FAIL    373
PASS     27
Name: count, dtype: int64

Top fail reasons:
fail_reason
Followers N outside N-N                          348
Engagement data unavailable or too few videos      9
Inactive: last upload N days ago                   7
Avg views N below N                                5
Engagement N.N% below N.N%                         3
Looks like a brand/official account                2
Not fashion/beauty content                         1
Name: count, dtype: int64


,name,followers,engagement_rate,niche,last_upload,flag
2,Skin Care Routine,11900,1.53,Beauty,2026-10-03,
85,Aanyaa,63700,4.11,Beauty,2026-10-02,
89,Makeup tutorial,5740,1.53,Beauty,2026-10-02,
107,Vijaya Haul Fashion's,27500,1.55,Fashion,2026-09-21,
145,Pranav,20400,1.59,Fashion,2026-10-03,
163,Dstyle - outfit ideas & outfit inspirations,25300,1.28,Fashion,2026-10-01,
184,@the_ outfit_ hub,20600,4.29,Fashion,2026-10-03,
255,Akansha Herbal,23700,2.17,Beauty,2026-09-28,
278,Himi Khandelwal,92000,5.26,Beauty,2026-09-25,
293,Ashi Beauty,94800,2.30,Beauty,2026-09-27,


27 shortlisted channels
Email found:     10
Instagram found: 2
Website found:   3


,name,email,instagram,content_themes
2,Skin Care Routine,Not Found,Not Found,"Skincare, Tutorials"
85,Aanyaa,aanyaaa1111@gmail.com,Not Found,"Makeup, Outfit/Styling, Tutorials"
89,Makeup tutorial,Not Found,Not Found,"Makeup, Tutorials"
107,Vijaya Haul Fashion's,Not Found,Not Found,Hauls
145,Pranav,pranaavbusinessinfo@gmail.com,Not Found,"Outfit/Styling, Hauls, Reviews"
163,Dstyle - outfit ideas & outfit inspirations,Not Found,https://www.instagram.com/dstyle_outfitideas/,Outfit/Styling
184,@the_ outfit_ hub,Not Found,Not Found,"Outfit/Styling, Tutorials"
255,Akansha Herbal,Not Found,Not Found,"Skincare, Haircare, Tutorials"
278,Himi Khandelwal,Not Found,Not Found,"Skincare, Haircare, Reviews"
293,Ashi Beauty,ashadilshi2000@gmail.com,Not Found,"Skincare, Haircare, Outfit/Styling, Reviews, T..."


In [7]:
!pip install -q google-genai

In [11]:
import json, time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-flash-latest"   # if this errors, check AI Studio for the current model name

# ---------- SAMPLE brand (fictional, label it as sample in the README) ----------
BRAND = {
    "name": "Glowly",
    "product": "a gentle vitamin C skincare range",
    "sender": "Harini",
}

def choose_angle(row):
    f, e = row["followers"], row["engagement_rate"]
    if f >= 60000:
        return "a paid sponsorship"
    if f >= 20000 or e >= 4:
        return "an affiliate campaign with a commission per sale"
    return "a free product barter collaboration with UGC content"

def wc(text):
    return len(text.split())

def build_prompt(row, angle, problem=""):
    titles = row["recent_titles"] if isinstance(row["recent_titles"], list) else []
    return f"""You write outreach messages for the brand "{BRAND['name']}" ({BRAND['product']}).
Use ONLY these facts about the creator. Do not invent details, numbers, or praise that is not supported.

Creator channel name: {row['name']}
Platform: YouTube
Niche: {row['niche']}
Recent video titles: {titles}
Proposed collaboration: {angle}

Return JSON with exactly these keys:
- "email_subject": short subject line
- "email_body": 60 to 90 words total, including greeting and sign-off ("{BRAND['sender']}, {BRAND['name']}"). Mention something specific from the recent video titles, the proposed collaboration, and the value for them.
- "instagram_dm": 15 to 30 words, casual and natural, personalized to their content.

Address them by the channel name as written; never guess a first name.
{problem}"""

def generate(row, angle, max_tries=3):
    problem = ""
    for attempt in range(max_tries):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=build_prompt(row, angle, problem),
                config=types.GenerateContentConfig(response_mime_type="application/json"),
            )
            data = json.loads(resp.text)
            e_wc, d_wc = wc(data["email_body"]), wc(data["instagram_dm"])
            if 60 <= e_wc <= 90 and 15 <= d_wc <= 30:
                return data, "OK", e_wc, d_wc
            problem = f"Previous attempt had email {e_wc} words and DM {d_wc} words. Fix the lengths."
        except Exception as ex:
            problem = ""
            print("retry", row["name"], "->", str(ex)[:80])
            time.sleep(5)
    return None, "FAILED_VALIDATION", None, None

out = []
for _, row in short.iterrows():
    angle = choose_angle(row)
    data, status, e_wc, d_wc = generate(row, angle)
    out.append({
        "channel_id": row["channel_id"],
        "collab_angle": angle,
        "email_subject": data["email_subject"] if data else "",
        "email_body": data["email_body"] if data else "",
        "instagram_dm": data["instagram_dm"] if data else "",
        "email_words": e_wc, "dm_words": d_wc,
        "generation_status": status,
        "review_status": "PENDING",
    })
    time.sleep(4)   # stay inside free-tier rate limits

msgs = short.merge(pd.DataFrame(out), on="channel_id")
print(msgs["generation_status"].value_counts())
msgs[["name", "collab_angle", "email_words", "dm_words", "generation_status"]]

retry Skin Care Routine  -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Skin Care Routine  -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Skin Care Routine  -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Aanyaa -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Aanyaa -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Aanyaa -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Makeup tutorial  -> 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
retry Vijaya Haul Fashion's -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Vijaya Haul Fashion's -> 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
retry Vijaya Haul Fashion's -> 503 UNAVAILABLE. {'e

,name,collab_angle,email_words,dm_words,generation_status
0,Skin Care Routine,a free product barter collaboration with UGC c...,NaN,NaN,FAILED_VALIDATION
1,Aanyaa,a paid sponsorship,NaN,NaN,FAILED_VALIDATION
2,Makeup tutorial,a free product barter collaboration with UGC c...,66.0,27.0,OK
3,Vijaya Haul Fashion's,an affiliate campaign with a commission per sale,NaN,NaN,FAILED_VALIDATION
4,Pranav,an affiliate campaign with a commission per sale,72.0,26.0,OK
5,Dstyle - outfit ideas & outfit inspirations,an affiliate campaign with a commission per sale,NaN,NaN,FAILED_VALIDATION
6,@the_ outfit_ hub,an affiliate campaign with a commission per sale,NaN,NaN,FAILED_VALIDATION
7,Akansha Herbal,an affiliate campaign with a commission per sale,NaN,NaN,FAILED_VALIDATION
8,Himi Khandelwal,a paid sponsorship,NaN,NaN,FAILED_VALIDATION
9,Ashi Beauty,a paid sponsorship,NaN,NaN,FAILED_VALIDATION


In [12]:
MODELS = ["gemini-flash-latest", "gemini-3.1-flash-lite", "gemini-2.5-flash-lite"]

def call_llm(prompt):
    """Try each model in turn; back off when the API is busy or rate-limited."""
    for model in MODELS:
        for wait in (10, 30, 60):
            try:
                resp = client.models.generate_content(
                    model=model, contents=prompt,
                    config=types.GenerateContentConfig(response_mime_type="application/json"),
                )
                return json.loads(resp.text), model
            except Exception as ex:
                msg = str(ex)
                if "PerDay" in msg or "per day" in msg.lower():
                    print(f"{model}: daily quota used up, trying next model")
                    break
                if "429" in msg or "503" in msg:
                    print(f"{model}: busy or rate-limited, waiting {wait}s")
                    time.sleep(wait)
                else:
                    print(f"{model}: error -> {msg[:100]}")
                    break
    return None, None

def generate2(row, angle):
    problem = ""
    for _ in range(3):
        data, used = call_llm(build_prompt(row, angle, problem))
        if data is None:
            return None, "API_UNAVAILABLE", None, None, None
        e, d = wc(data["email_body"]), wc(data["instagram_dm"])
        if 60 <= e <= 90 and 15 <= d <= 30:
            return data, "OK", e, d, used
        problem = f"Previous attempt had email {e} words and DM {d} words. Fix the lengths."
    return None, "FAILED_VALIDATION", None, None, None

# keep what already worked (from the earlier run), redo the rest
done = {r["channel_id"]: r for r in out if r["generation_status"] == "OK"}

for _, row in short.iterrows():
    cid = row["channel_id"]
    if cid in done and done[cid]["generation_status"] == "OK":
        continue
    angle = choose_angle(row)
    data, status, e, d, used = generate2(row, angle)
    done[cid] = {
        "channel_id": cid, "collab_angle": angle,
        "email_subject": data["email_subject"] if data else "",
        "email_body": data["email_body"] if data else "",
        "instagram_dm": data["instagram_dm"] if data else "",
        "email_words": e, "dm_words": d,
        "generation_status": status, "model_used": used, "review_status": "PENDING",
    }
    print(f"{row['name']}: {status}")
    if status == "API_UNAVAILABLE":
        print("All models unavailable right now. Stopping early; rerun this cell later.")
        break
    time.sleep(8)

msgs = short.merge(pd.DataFrame(list(done.values())), on="channel_id", how="left")
msgs["generation_status"] = msgs["generation_status"].fillna("NOT_RUN")
print(msgs["generation_status"].value_counts())

gemini-flash-latest: daily quota used up, trying next model
Skin Care Routine : OK
gemini-flash-latest: daily quota used up, trying next model
Aanyaa: OK
gemini-flash-latest: daily quota used up, trying next model
Vijaya Haul Fashion's: OK
gemini-flash-latest: daily quota used up, trying next model
Dstyle - outfit ideas & outfit inspirations: OK
gemini-flash-latest: daily quota used up, trying next model
@the_ outfit_ hub: OK
gemini-flash-latest: daily quota used up, trying next model
Akansha Herbal: OK
gemini-flash-latest: daily quota used up, trying next model
Himi Khandelwal: OK
gemini-flash-latest: daily quota used up, trying next model
Ashi Beauty: OK
gemini-flash-latest: daily quota used up, trying next model
Style Tips: OK
gemini-flash-latest: daily quota used up, trying next model
Fashion Tips With Moon: OK
gemini-flash-latest: daily quota used up, trying next model
Rajni Style Tips: OK
gemini-flash-latest: daily quota used up, trying next model
Style Alchemy By Priyanshi : OK


In [13]:
msgs["model_used"] = msgs["model_used"].fillna("gemini-flash-latest")  # the 2 earlier ones

for _, r in msgs.sample(4, random_state=1).iterrows():
    print("TO:", r["name"], "|", r["email"], "|", r["collab_angle"])
    print("RECENT VIDEOS:", r["recent_titles"])
    print("SUBJECT:", r["email_subject"])
    print(r["email_body"], "\n")
    print("DM:", r["instagram_dm"], "\n" + "-" * 60)

TO: Rasteena's Style | rasteena78@gmail.com | an affiliate campaign with a commission per sale
RECENT VIDEOS: ["Rasteena's Diary Vlog #03 #minivlog #dailyvlog #gymlife #shorts #youtubeshorts #vlogging", "Rasteena's Diary Vlog #02 #minivlog #dailyvlog #vlog #lifeupdates #yoitubeshorts", "Rasteena's Diary Vlog #01 #minivlog #dailyvlogs #workout #vlog #shorts #ytshorts #youtubevloggers"]
SUBJECT: Collaboration inquiry: Rasteena's Style x Glowly
Hi Rasteena's Style, I’ve been following your YouTube channel and really enjoyed your recent 'Rasteena's Diary Vlog' series. Your daily vlogs capture such an authentic energy that aligns perfectly with our brand, Glowly. We are a gentle vitamin C skincare range and would love to invite you to join our affiliate campaign. This is a great way to earn a commission for every sale you generate while sharing products you genuinely love with your audience. Would you be interested in learning more? Harini, Glowly. 

DM: Hey Rasteena's Style! I love your da

In [14]:
import re

# ---------- 1. Review flags ----------
SELLER_WORDS = ["clearance", "free shipping", "live offers", "wholesale", "new stock", "/-", "cod available", "dm to order"]
NICHE_FLAT = [k for kws in NICHE_KEYWORDS.values() for k in kws]

def titles_of(row):
    return row["recent_titles"] if isinstance(row["recent_titles"], list) else []

def is_seller(row):
    text = (" ".join(titles_of(row)) + " " + row["description"][:300]).lower()
    return sum(w in text for w in SELLER_WORDS) >= 2

def non_latin_share(row):
    letters = [c for t in titles_of(row) for c in t if c.isalpha()]
    return sum(ord(c) > 591 for c in letters) / len(letters) if letters else 0

def niche_in_titles(row):
    text = " ".join(titles_of(row)).lower()
    return any(k in text for k in NICHE_FLAT)

def make_flags(row):
    flags = []
    if is_seller(row):
        flags.append("Looks like a seller/shop, not a creator")
    if non_latin_share(row) > 0.4:
        flags.append("Non-English content: check language/audience fit")
    if not niche_in_titles(row):
        flags.append("Recent videos don't show fashion/beauty content")
    if row["email"] == "Not Found" and row["instagram"] == "Not Found":
        flags.append("No contact route found")
    return flags

flag_rows = []
for _, r in short.iterrows():
    fl = make_flags(r)
    if any("seller" in f for f in fl):
        status = "REJECTED"
    elif fl:
        status = "NEEDS_MANUAL_CHECK"
    else:
        status = "PENDING"
    flag_rows.append({"channel_id": r["channel_id"], "review_flags": "; ".join(fl), "review_status": status})
flags = pd.DataFrame(flag_rows)

# ---------- 2. Stricter prompt (replaces the old build_prompt) ----------
def build_prompt(row, angle, problem=""):
    return f"""You write outreach messages for the brand "{BRAND['name']}" ({BRAND['product']}).
Use ONLY these facts about the creator. Do not invent details, numbers, or praise that is not supported.

Creator channel name: {row['name']}
Platform: YouTube
Niche: {row['niche']}
Recent video titles: {titles_of(row)}
Proposed collaboration: {angle}

Return JSON with exactly these keys:
- "email_subject": short subject line
- "email_body": 60 to 90 words total, including greeting and sign-off ("{BRAND['sender']}, {BRAND['name']}"). Mention one specific recent video title and the proposed collaboration.
- "instagram_dm": 15 to 30 words, casual and natural.

Strict rules:
- Start the email with "Hi there," and the DM with "Hi!". Never use the channel name as a greeting.
- Never claim to have followed, subscribed to, watched regularly, loved, or enjoyed anything. Say "I came across your video ..." instead.
- No flattery or claims about their audience that the titles do not support.
- Do not say their content "aligns perfectly" with the brand.
{problem}"""

# ---------- 3. Regenerate everything except rejected ----------
rejected_ids = set(flags[flags.review_status == "REJECTED"].channel_id)
for _, row in short.iterrows():
    cid = row["channel_id"]
    if cid in rejected_ids:
        continue
    angle = choose_angle(row)
    data, status, e, d, used = generate2(row, angle)
    done[cid] = {
        "channel_id": cid, "collab_angle": angle,
        "email_subject": data["email_subject"] if data else "",
        "email_body": data["email_body"] if data else "",
        "instagram_dm": data["instagram_dm"] if data else "",
        "email_words": e, "dm_words": d,
        "generation_status": status, "model_used": used,
    }
    print(f"{row['name']}: {status}")
    if status == "API_UNAVAILABLE":
        print("Models unavailable (probably daily quota). Stopping; rerun this cell later.")
        break
    time.sleep(8)

# ---------- 4. Rebuild the table ----------
gen = pd.DataFrame(list(done.values())).drop(columns=["review_status"], errors="ignore")
msgs = short.merge(gen, on="channel_id", how="left").merge(flags, on="channel_id", how="left")
msgs["generation_status"] = msgs["generation_status"].fillna("NOT_RUN")
print(msgs["review_status"].value_counts())
msgs[["name", "review_status", "review_flags"]]

gemini-flash-latest: daily quota used up, trying next model
Skin Care Routine : OK
gemini-flash-latest: daily quota used up, trying next model
Aanyaa: OK
gemini-flash-latest: daily quota used up, trying next model
Makeup tutorial : OK
gemini-flash-latest: daily quota used up, trying next model
Pranav: OK
gemini-flash-latest: daily quota used up, trying next model
Dstyle - outfit ideas & outfit inspirations: OK
gemini-flash-latest: daily quota used up, trying next model
@the_ outfit_ hub: OK
gemini-flash-latest: daily quota used up, trying next model
Akansha Herbal: OK
gemini-flash-latest: daily quota used up, trying next model
Himi Khandelwal: OK
gemini-flash-latest: daily quota used up, trying next model
Ashi Beauty: OK
gemini-flash-latest: daily quota used up, trying next model
Style Tips: OK
gemini-flash-latest: daily quota used up, trying next model
Fashion Tips With Moon: OK
gemini-flash-latest: daily quota used up, trying next model
Rajni Style Tips: OK
gemini-flash-latest: daily

,name,review_status,review_flags
0,Skin Care Routine,NEEDS_MANUAL_CHECK,No contact route found
1,Aanyaa,PENDING,
2,Makeup tutorial,NEEDS_MANUAL_CHECK,No contact route found
3,Vijaya Haul Fashion's,REJECTED,"Looks like a seller/shop, not a creator; Recen..."
4,Pranav,PENDING,
5,Dstyle - outfit ideas & outfit inspirations,PENDING,
6,@the_ outfit_ hub,NEEDS_MANUAL_CHECK,No contact route found
7,Akansha Herbal,NEEDS_MANUAL_CHECK,Non-English content: check language/audience f...
8,Himi Khandelwal,NEEDS_MANUAL_CHECK,Recent videos don't show fashion/beauty conten...
9,Ashi Beauty,NEEDS_MANUAL_CHECK,Non-English content: check language/audience fit


In [15]:
BANNED = ["following your", "been following", "really enjoyed", "loved your",
          "aligns perfectly", "subscribed", "big fan"]

def rule_violations(r):
    if r["generation_status"] != "OK":
        return ""
    text = f"{r['email_body']} {r['instagram_dm']}".lower()
    hits = [b for b in BANNED if b in text]
    if not r["email_body"].lower().startswith("hi there"):
        hits.append("wrong greeting")
    return ", ".join(hits)

# rejected channels shouldn't carry any message
msgs.loc[msgs.review_status == "REJECTED", ["email_subject", "email_body", "instagram_dm"]] = ""

msgs["rule_violations"] = msgs.apply(rule_violations, axis=1)
print(msgs["generation_status"].value_counts())
print((msgs["rule_violations"] != "").sum(), "messages still break the rules")
msgs[msgs["rule_violations"] != ""][["name", "rule_violations"]]

generation_status
OK    27
Name: count, dtype: int64
1 messages still break the rules


,name,rule_violations
3,Vijaya Haul Fashion's,wrong greeting


In [16]:
msgs.loc[msgs.review_status == "REJECTED", "generation_status"] = "SKIPPED_REJECTED"
msgs["rule_violations"] = msgs.apply(rule_violations, axis=1)
print((msgs["rule_violations"] != "").sum(), "messages break the rules")

0 messages break the rules


In [17]:
from google.colab import files
msgs.drop(columns=["description"], errors="ignore").to_csv("influencers_with_messages.csv", index=False)
files.download("influencers_with_messages.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
ready = msgs[(msgs.review_status == "PENDING") & (msgs.generation_status == "OK")]
print(len(ready), "messages to review\n")
for _, r in ready.iterrows():
    print("TO:", r["name"], "|", r["email"], "|", r["instagram"], "|", r["collab_angle"])
    print("RECENT VIDEOS:", r["recent_titles"])
    print("SUBJECT:", r["email_subject"])
    print(r["email_body"], "\n")
    print("DM:", r["instagram_dm"], "\n" + "-" * 60)

8 messages to review

TO: Aanyaa | aanyaaa1111@gmail.com | Not Found | a paid sponsorship
RECENT VIDEOS: ['use my code Aanya12 for upto 20% off', 'use my code Aanya12 for upto 20% off', '#tuscanmakeup']
SUBJECT: Paid collaboration opportunity with Glowly
Hi there, I came across your video titled #tuscanmakeup and wanted to reach out regarding a potential collaboration. I represent Glowly, a gentle vitamin C skincare range, and we are interested in working with you on a paid sponsorship for our products. Please let us know if you are open to discussing this opportunity further and if you have a media kit available for review. We look forward to hearing from you soon. Best regards, Harini, Glowly. 

DM: Hi! I came across your #tuscanmakeup video and would love to discuss a paid sponsorship opportunity with our skincare brand, Glowly. Are you interested in collaborating? 
------------------------------------------------------------
TO: Pranav | pranaavbusinessinfo@gmail.com | Not Found | 

In [19]:
names = msgs["name"].str.strip()

APPROVE = ["Aanyaa", "Pranav", "Dstyle - outfit ideas & outfit inspirations",
           "Rajni Style Tips", "Littlegunnu1", "Mrs. Charity Umar Natural Hair Care"]
REJECT = {"Style Tips": "Recent videos are religious vlogs, not fashion/beauty",
          "Sofia Hair Health": "Hair-loss clinic/brand account, poor product fit"}

msgs.loc[names.isin(APPROVE) & (msgs.review_status == "PENDING"), "review_status"] = "APPROVED"
for n, why in REJECT.items():
    msgs.loc[names == n, "review_status"] = "REJECTED"
    msgs.loc[names == n, "review_flags"] = why

print(msgs.review_status.value_counts())

review_status
NEEDS_MANUAL_CHECK    18
APPROVED               6
REJECTED               3
Name: count, dtype: int64


In [20]:
import sqlite3, re, smtplib, datetime
from email.message import EmailMessage

DRY_RUN = True            # keep True: Glowly is a fictional brand
TEST_RECIPIENT = ""       # your own email; only used if DRY_RUN is False

conn = sqlite3.connect("outreach.db")
conn.execute("""CREATE TABLE IF NOT EXISTS outreach_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    influencer TEXT, email TEXT UNIQUE, channel_id TEXT,
    sent TEXT, sent_at TEXT, status TEXT, detail TEXT)""")
conn.execute("""CREATE TABLE IF NOT EXISTS dm_queue (
    channel_id TEXT PRIMARY KEY, influencer TEXT, instagram TEXT,
    dm_text TEXT, status TEXT)""")
conn.execute("DELETE FROM outreach_log")   # start this demo run clean
conn.execute("DELETE FROM dm_queue")
conn.commit()

EMAIL_OK = re.compile(r"^[^@\s]+@[^@\s]+\.[A-Za-z]{2,}$")

def send_email(to, subject, body):
    if DRY_RUN:
        return "NO", "SIMULATED", "dry run: nothing sent"
    try:
        from google.colab import userdata
        msg = EmailMessage()
        msg["From"] = userdata.get("GMAIL_ADDRESS")
        msg["To"] = TEST_RECIPIENT
        msg["Subject"] = f"[TEST for {to}] {subject}"
        msg.set_content(body)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
            s.login(userdata.get("GMAIL_ADDRESS"), userdata.get("GMAIL_APP_PASSWORD"))
            s.send_message(msg)
        return "TEST_INBOX", "SENT_TO_TEST_INBOX", "redirected to test recipient"
    except Exception as e:
        return "NO", "FAILED", str(e)[:150]

def run_sending(df):
    for _, r in df.iterrows():
        if r["review_status"] != "APPROVED":
            continue
        ig = r["instagram"]
        conn.execute("INSERT OR IGNORE INTO dm_queue VALUES (?,?,?,?,?)",
            (r["channel_id"], r["name"], ig, r["instagram_dm"],
             "MANUAL_SEND_REQUIRED" if ig != "Not Found" else "NO_INSTAGRAM_HANDLE_FOUND"))
        email = str(r["email"]).lower()
        if not EMAIL_OK.match(email):
            continue
        if conn.execute("SELECT 1 FROM outreach_log WHERE email=?", (email,)).fetchone():
            print("skipped duplicate:", email)
            continue
        sent, status, detail = send_email(email, r["email_subject"], r["email_body"])
        conn.execute(
            "INSERT INTO outreach_log (influencer,email,channel_id,sent,sent_at,status,detail) VALUES (?,?,?,?,?,?,?)",
            (r["name"], email, r["channel_id"], sent,
             datetime.datetime.now().isoformat(timespec="seconds"), status, detail))
        conn.commit()
        print(status, "->", r["name"])
    conn.commit()

print("--- first run ---")
run_sending(msgs)
print("--- second run (should skip every email) ---")
run_sending(msgs)

--- first run ---
SIMULATED -> Aanyaa
SIMULATED -> Pranav
SIMULATED -> Rajni Style Tips
SIMULATED -> Littlegunnu1
SIMULATED -> Mrs. Charity Umar Natural Hair Care
--- second run (should skip every email) ---
skipped duplicate: aanyaaa1111@gmail.com
skipped duplicate: pranaavbusinessinfo@gmail.com
skipped duplicate: rajnichaudhary1908@gmail.com
skipped duplicate: business-littlegunnu1@gmail.com
skipped duplicate: unveilingtheveilhair@gmail.com


In [21]:
from google.colab import files

log = pd.read_sql("SELECT channel_id, sent, sent_at, status FROM outreach_log", conn)
t = msgs[["name", "email", "channel_id", "generation_status", "review_status"]].merge(log, on="channel_id", how="left")

def final_status(r):
    if pd.notna(r["status"]):
        return r["status"]
    if r["review_status"] == "REJECTED":
        return "REJECTED"
    if r["review_status"] == "NEEDS_MANUAL_CHECK":
        return "HELD_FOR_REVIEW"
    if r["email"] == "Not Found":
        return "NO_EMAIL_FOUND"
    return "NOT_SENT"

t["Status"] = t.apply(final_status, axis=1)
t["Sent"] = t["sent"].fillna("NO")
t["Message Generated"] = t["generation_status"].map(lambda s: "Yes" if s == "OK" else "No")
tracker = t.rename(columns={"name": "Influencer", "email": "Email", "sent_at": "Date"})[
    ["Influencer", "Email", "Message Generated", "Sent", "Date", "Status"]]

dm = pd.read_sql("SELECT * FROM dm_queue", conn)

tracker.to_csv("outreach_tracker.csv", index=False)
dm.to_csv("instagram_dm_queue.csv", index=False)
files.download("outreach_tracker.csv")
files.download("instagram_dm_queue.csv")

print(tracker["Status"].value_counts())
print()
print(dm[["influencer", "status"]])

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Status
HELD_FOR_REVIEW    18
SIMULATED           5
REJECTED            3
NO_EMAIL_FOUND      1
Name: count, dtype: int64

                                    influencer                     status
0                                       Aanyaa  NO_INSTAGRAM_HANDLE_FOUND
1                                       Pranav  NO_INSTAGRAM_HANDLE_FOUND
2  Dstyle - outfit ideas & outfit inspirations       MANUAL_SEND_REQUIRED
3                             Rajni Style Tips       MANUAL_SEND_REQUIRED
4                                 Littlegunnu1  NO_INSTAGRAM_HANDLE_FOUND
5          Mrs. Charity Umar Natural Hair Care  NO_INSTAGRAM_HANDLE_FOUND


In [9]:
for m in client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [10]:
for MODEL in ["gemini-flash-latest", "gemini-3.1-flash-lite", "gemini-2.5-flash-lite"]:
    try:
        resp = client.models.generate_content(model=MODEL, contents="Say hello in 5 words.")
        print("WORKS:", MODEL, "->", resp.text.strip())
        break
    except Exception as e:
        print("FAILED:", MODEL, "->", str(e)[:200])

WORKS: gemini-flash-latest -> Hello, how are you today?
